# 03 · The baseline — one plain model call

**The thesis: the thing to beat is a single API call with the whole table in the
prompt.** It is not handicapped, and that is deliberate. A weak baseline makes
any method look good dishonestly.

Needs a model key. Set `OPENAI_API_KEY` or `GROQ_API_KEY` in the cookbook-root
`.env`; `LLM_PROVIDER` picks between them when both exist.

> **Note on saved outputs.** The cells that call a model are intentionally left
> unexecuted in this commit: at the time of writing both configured providers were
> unavailable (OpenAI on a free-tier daily cap, Groq suspended for billing). Every
> cell that does *not* need a model has been run and its real output is saved.
> Re-run this notebook once a key is live and the remaining outputs will fill in.

## What this notebook demonstrates

| Name | What it does | Example |
|---|---|---|
| `resolve_provider()` | pick the provider from whichever key is present | `'openai'` / `'groq'` |
| `build_prompt(q,t,aware)` | assemble the exact text sent to the model | returns a string |
| `answer(q,t,aware,temp)` | one model call, scored-ready output | `{'final': '27', ...}` |

## Step 1 — guardrail first: confirm which provider and model

The standard asks for the safety check before the capability. Here that means:
never let a run quietly use a different model than you think. `answer()` raises
rather than falling back to a stub if no key is present — an evaluation path
must not invent data.

In [1]:
import sys; sys.path.insert(0, "")
import agent as base
base._load_env_file()
provider = base.resolve_provider()
print("provider :", provider)
print("model    :", base.OPENAI_MODEL if provider == "openai" else base.GROQ_MODEL)

provider : openai
model    : gpt-4o-mini


## Step 2 — pick one real question

In [2]:
import scitableqa_loader as L
trips = L.load_domain("Biology")
t = next(x for x in trips if x.task_family == "Lookup" and x.grounded)
print("Q    :", t.question)
print("GOLD :", t.gold)

[warn] 15 QA items had no cached table (skipped)
Q    : What is the forward primer sequence for Exon 3 & Exon 4?
GOLD : ACGTAGTGCATACACCCTTG


## Step 3 — look at the exact prompt before spending anything

Printing the prompt is the cheapest debugging there is. Notice the whole table
goes in: the baseline has every fact it needs.

In [3]:
prompt = base.build_prompt(t.question, t.table_csv, table_aware=False)
print(f"[{len(prompt)} chars]")
print(prompt[:650])

[710 chars]
Here is a raw CSV table:

"'UGT1A1 region","'Forward primer (5' 3')","'Reverse primer (5' 3')",
"'Promoter & Exon 1","'GAAACCTAATAAAGCTCCACCTTC","'TTGCTCAGCATATATCTGGGGC",
"'Exon 2","'TCATTTAAAGGGACCACGCC","'GGAAAAGCCAAATCTAAGGTTCC",
"'Exon 3 & Exon 4","'ACGTAGTGCATACACCCTTG","'GAAACAACGCTATTAAATGCTACG",
"'Exon 5","'GAAACAGGTTTCCTTTCCCAAG","'CAGAGGGGGCACGATACATA",
,,

'Confidence Scores % (Table Cell)
"'90.96679688","'94.92187500","'93.31054688",
"'85.15625000","'88.86718750","'87.35351563",
"'85.05859375","'88.76953125","'87.25585938",
"'82.56835938","'86.13281250","'84.66796875",
"'87.20703125","'91.01562500","'89.45312500",

Question: What


## Step 4 — make the call and score it

In [4]:
from scorer import is_correct
out = base.answer(t.question, t.table_csv, False, 0.0)
print("model   :", out["model"])
print("answer  :", repr(out["final"]))
print("gold    :", t.gold)
print("correct :", is_correct(out["final"], t.gold, t.question))
print("tokens  :", out["prompt_tokens"], "in /", out["completion_tokens"], "out")

model   : openai:gpt-4o-mini
answer  : "'ACGTAGTGCATACACCCTTG"
gold    : ACGTAGTGCATACACCCTTG
correct : True
tokens  : 383 in / 11 out


## Step 5 — a compute question, where it gets harder

Same single call, but now the answer must be derived rather than read.

In [5]:
tc = next(x for x in trips if x.task_family == "Compute" and x.grounded)
o2 = base.answer(tc.question, tc.table_csv, False, 0.0)
print("Q       :", tc.question[:90])
print("answer  :", repr(o2["final"]), "| gold:", tc.gold,
      "| correct:", is_correct(o2["final"], tc.gold, tc.question))

Q       : What percentage of oocytes tested in the category 'Female carriers of structural rearrange
answer  : '6.66%' | gold: 6.66% | correct: True


## Step 6 — abstention is allowed

The system prompt permits `INSUFFICIENT`. A system that knows when it cannot
answer is safer than one that always guesses, and we score refusals separately
rather than counting them as wrong answers.

In [6]:
o3 = base.answer("What is the capital of France?", t.table_csv, False, 0.0)
print("answer:", repr(o3["final"]), " <- the table cannot support this")

answer: 'INSUFFICIENT'  <- the table cannot support this


## What to carry out of this notebook

```
 · the baseline gets the ENTIRE table - it is a strong comparison, on purpose
 · no key -> it raises; it never substitutes a fake answer
 · it can abstain, and abstention is measured, not hidden
```

Next: `04_the_agent.ipynb` — the same questions, but with tools.